In [1]:
import os
os.chdir('/workspace/f839811a-5d45-4a51-8681-5f83eaad18f6')
os.makedirs('cached_zeros/zeta', exist_ok=True)
os.makedirs('cached_zeros/Lchi5', exist_ok=True)
os.makedirs('downloads', exist_ok=True)
!ls -la


total 54
drwxrwsr-x 7 root jupytergroup 4096 May 30 17:32 .
drwxr-xr-x 5 root root 4096 May 30 17:32 ..
drwxrwsr-x 5 root jupytergroup 4096 May 30 17:32 .config
-rw-rw-r-- 1 kerneluser jupytergroup 0 May 30 17:32 .kernel_llm_logs_1.txt
drwxrwsr-x 2 root jupytergroup 4096 May 30 17:32 .prompts
-rw-r--r-- 1 root jupytergroup 25848 May 30 17:32 -PROMPT-v6-DATASET.md
drwxrwsr-x 4 kerneluser jupytergroup 4096 May 30 17:32 cached_zeros
drwxrwsr-x 2 kerneluser jupytergroup 4096 May 30 17:32 downloads
drwxr-sr-x 2 root jupytergroup 4096 May 30 17:32 memory


In [2]:

# Plan:
# 1. Download Odlyzko zeros6.gz (or smaller file with 20k+ zeros for our needs). 
# The actual file "zeros6.gz" from Odlyzko is huge. There's a smaller "zeros1" file with 100k zeros.
# For our needs (max 20000 zeros), we can use Odlyzko's "zeros1" file which contains 100,000 zeros.
# 2. For LMFDB L(chi_4 mod 5), download zeros.
# 3. Validate R2 gate.
# 4. Refine to dps=50, 80 with mpmath.findroot.

# First check internet & try the Odlyzko zeros1 file (100k zeros, ~1MB)
import urllib.request
import time

# Try Odlyzko's zeros1 first (smaller, contains 100k zeros which is plenty)
url_zeros1 = "https://www-users.cse.umn.edu/~odlyzko/zeta_tables/zeros1"

try:
 t0 = time.time()
 urllib.request.urlretrieve(url_zeros1, "downloads/odlyzko_zeros1.txt")
 print(f"Downloaded zeros1 in {time.time()-t0:.1f}s")
 print(f"Size: {os.path.getsize('downloads/odlyzko_zeros1.txt')} bytes")
except Exception as e:
 print(f"Failed: {e}")


Downloaded zeros1 in 0.7s
Size: 1800000 bytes


In [3]:

# Inspect zeros1
with open('downloads/odlyzko_zeros1.txt') as f:
 head = [next(f) for _ in range(5)]
print(head)
# count lines
import subprocess
r = subprocess.run(['wc', '-l', 'downloads/odlyzko_zeros1.txt'], capture_output=True, text=True)
print(r.stdout)


[' 14.134725142\n', ' 21.022039639\n', ' 25.010857580\n', ' 30.424876126\n', ' 32.935061588\n']
100000 downloads/odlyzko_zeros1.txt



In [4]:

# Excellent: 100,000 zeros, ~9-10 dp precision. Matches the R2 gate.
# Now validate R2: First three zeros should match 14.134725141, 21.022039639, 25.010857580 to >=6 dp.

import mpmath as mp

with open('downloads/odlyzko_zeros1.txt') as f:
 odlyzko_zeros = [mp.mpf(line.strip()) for line in f if line.strip()]

print(f"Total zeros: {len(odlyzko_zeros)}")
print(f"First 3: {odlyzko_zeros[:3]}")

r2_targets = [mp.mpf('14.134725141'), mp.mpf('21.022039639'), mp.mpf('25.010857580')]
for i, (got, want) in enumerate(zip(odlyzko_zeros[:3], r2_targets)):
 diff = abs(got - want)
 print(f" γ_{i+1}: got={got}, target={want}, |diff|={diff}, pass(>=1e-6)={diff < mp.mpf('1e-6')}")


Total zeros: 100000
First 3: [mpf('14.134725142000001'), mpf('21.022039638999999'), mpf('25.01085758')]
 γ_1: got=14.134725142, target=14.134725141, |diff|=1.00000008274037e-9, pass(>=1e-6)=True
 γ_2: got=21.022039639, target=21.022039639, |diff|=0.0, pass(>=1e-6)=True
 γ_3: got=25.01085758, target=25.01085758, |diff|=0.0, pass(>=1e-6)=True


In [5]:

# Great. R2 gate passes for zeta zeros.

# Now LMFDB L(chi_4 mod 5). The notation: chi mod 5, with "4" being the orbit / character label.
# In LMFDB, Dirichlet character is labeled "5.4" -> conductor 5, label 4.
# The non-trivial primitive character mod 5 is the unique real one (order 2): 5.4.
# Or order 4: 5.2, 5.3. There are 4 non-trivial characters mod 5.
# Looking at the prompt: "chi_4 mod 5" - this is the Dirichlet character 5.4 in LMFDB notation
# which is the real quadratic character (the Kronecker/Legendre symbol mod 5).
# L-function: 1.5.4.a.a -- actually for chi 5.4 the L-function label is "1-5-4.a-r0-0-0" etc.

# Better: try LMFDB API for L-function zeros.
# We can use LMFDB's L-function for label 1-5-4.b.a (for chi 5.4)
# Search LMFDB: https://www.lmfdb.org/L/Character/Dirichlet/5/4

# Try LMFDB API
import urllib.request, json
url = "https://www.lmfdb.org/api/lfunc_lfunctions/?conductor=5&_format=json&_fields=label,zeros,first_zero&_limit=20"
try:
 req = urllib.request.Request(url, headers={'User-Agent':'Mozilla/5.0'})
 resp = urllib.request.urlopen(req, timeout=30)
 data = json.loads(resp.read())
 print(json.dumps(data, indent=2)[:3000])
except Exception as e:
 print(f"Failed: {e}")


Failed: HTTP Error 404: Not Found


In [6]:

# Try direct LMFDB page
url = "https://www.lmfdb.org/L/1/5/4.b/a/0/0"
try:
 req = urllib.request.Request(url, headers={'User-Agent':'Mozilla/5.0'})
 resp = urllib.request.urlopen(req, timeout=30)
 html = resp.read().decode('utf-8', errors='ignore')
 print(len(html))
 # search for "zeros"
 idx = html.lower().find('zeros')
 print(html[max(0,idx-200):idx+2000])
except Exception as e:
 print(f"Failed: {e}")


Failed: HTTP Error 404: Not Found


In [7]:

# LMFDB direct access is often blocked / changing URLs. The dataset description warns about this:
# "direct automated downloads from LMFDB are often blocked".
# For L(chi_4 mod 5), we can compute zeros using mpmath. The character chi_4 mod 5 is the
# real (quadratic) Dirichlet character: chi(1)=1, chi(2)=-1, chi(3)=-1, chi(4)=1, chi(5)=0.
# mpmath has Lfunc -- actually, we can use mpmath.dirichlet or define L via dirichlet series.

# mpmath.dirichlet exists. Let me check.
import mpmath as mp
help(mp.dirichlet)


Help on method dirichlet in module mpmath.functions.zeta:

dirichlet(s, chi=[1], derivative=0) method of mpmath.ctx_mp.MPContext instance
 Evaluates the Dirichlet L-function

 .. math ::

 L(s,\chi) = \sum_{k=1}^\infty \frac{\chi(k)}{k^s}.

 where `\chi` is a periodic sequence of length `q` which should be supplied
 in the form of a list `[\chi(0), \chi(1), \ldots, \chi(q-1)]`.
 Strictly, `\chi` should be a Dirichlet character, but any periodic
 sequence will work.

 For example, ``dirichlet(s, [1])`` gives the ordinary
 Riemann zeta function and ``dirichlet(s, [-1,1])`` gives
 the alternating zeta function (Dirichlet eta function).

 Also the derivative with respect to `s` (currently only a first
 derivative) can be evaluated.

 **Examples**

 The ordinary Riemann zeta function::

 >>> from mpmath import *
 >>> mp.dps = 25; mp.pretty = True
 >>> dirichlet(3, [1]); zeta(3)
 1.202056903159594285399738
 1.202056903159594285399738
 >>> dirichlet(1, [1])
 +inf

 The alternating zeta functi

In [8]:

# Good. For chi_4 mod 5, the character is the real quadratic one.
# In LMFDB numbering, the Dirichlet character 5.4 -- since (Z/5Z)* is cyclic of order 4,
# generator g=2 (2^1=2, 2^2=4, 2^3=3, 2^4=1). The character "5.4" sends 2 to e^{2*pi*i*3/4} typically.
# But the literal "chi_4 mod 5" referenced in dataset usually means the unique real non-trivial,
# i.e., the Legendre symbol mod 5: chi(1)=1, chi(2)=-1, chi(3)=-1, chi(4)=1, chi(5)=0.
# Period of length q=5 in mpmath.dirichlet: chi at position k for k=0..q-1.
# k=0: chi(0)=0 (or chi(5)=0)
# k=1: 1
# k=2: -1
# k=3: -1
# k=4: 1

chi5 = [0, 1, -1, -1, 1]
print("L(2, chi):", mp.dirichlet(2, chi5))
# Check value at s=1: should be pi/something (real character, L(1,chi)=2*pi/sqrt(20) or similar)
print("L(1, chi):", mp.dirichlet(1, chi5))
# L(1,chi) for chi = Legendre(./5) = pi / sqrt(5) * (some). Actually for real chi mod p, L(1) = pi h /sqrt(p)? 
# Just trust the function.

# To find zeros, use Hardy Z-function approach. The completed L-function:
# Lambda(s, chi) = (5/pi)^{s/2} Gamma((s+a)/2) L(s, chi)
# For odd character chi (chi(-1)=-1), a=1. For even character (chi(-1)=1), a=0.
# chi(-1)=chi(4)=1, so this is EVEN character. a=0.
# Functional equation: Lambda(s,chi) = W * Lambda(1-s, conj(chi))
# Since chi is real, conj(chi)=chi.

# The Hardy-like Z function for L(s,chi) on critical line:
# Z(t) = exp(i*theta(t)) * L(1/2 + i*t, chi)
# where theta is the phase such that Z is real.

# But for finding zeros, simplest: use mpmath's siegelz or build via Lambda function.
# Actually we just need zeros. Let's find them via L(1/2 + i*t, chi) imag/real sign changes.

# Let me compute L on the critical line at first few t to see where zeros are.
def Lfun(t):
 return mp.dirichlet(mp.mpf('0.5') + 1j*t, chi5)

mp.mp.dps = 30
for t in [1, 5, 6, 6.5, 7, 8, 10, 12, 14, 15]:
 val = Lfun(t)
 print(f"t={t}: L = {val}")


L(2, chi): 0.706211403259741
L(1, chi): 0.430408940964004
t=1: L = (0.31563908354458890802148708156 + 0.453571197449460288056190547813j)
t=5: L = (1.80322202725409466062731735527 - 1.14053793770687586260151661422j)
t=6: L = (0.275543455389521803395512886745 - 0.995392028773643947872231871833j)
t=6.5: L = (-0.031690135090975650494724353413 - 0.241187000196791614246747033849j)
t=7: L = (0.292323361306636580383315496887 + 0.47611508639377517670306371291j)
t=8: L = (1.59438209184126173803110725339 + 0.204677103986502754635956698147j)
t=10: L = (0.0700622754898766549129240431471 + 0.25541118768749353716404978985j)
t=12: L = (-0.060751952094698386832026332548 + 0.071235106909034033757249137691j)
t=14: L = (4.24158395996044573641450151483 - 0.247729709280757745183561993076j)
t=15: L = (0.793352044357111652360097023719 - 2.66050203243586399001664023229j)


In [9]:

# Issue: L on critical line is complex (zeros are isolated points where L=0).
# For approach to find zeros, I need a real-valued Z-function.
# For chi mod 5 (q=5), even (chi(-1)=1, a=0):
# Lambda(s) = (q/pi)^{s/2} Gamma(s/2) L(s, chi)
# Functional eq: Lambda(s) = W * Lambda(1-s)
# For real chi: W = tau(chi)/sqrt(q) and is real (= +1 typically for principal real chi).
# tau(chi) for Legendre symbol mod 5: tau(chi) = sum_{k=1}^4 chi(k) e^{2*pi*i*k/5}
# = e^(2pi i/5) - e^(4pi i/5) - e^(6pi i/5) + e^(8pi i/5)
# For p ≡ 1 mod 4, tau = sqrt(p). So W = sqrt(5)/sqrt(5) = 1.
# So Z(t) := pi^{-it/2} q^{it/2} Gamma((1/2+it)/2)/|Gamma((1/2+it)/2)| L(1/2+it, chi)
# is real.

# Use theta function: theta(t) = arg( (q/pi)^{it/2} Gamma((1/2+it)/2) ) = (t/2) ln(q/pi) + arg(Gamma((1/2+it)/2))
# Z(t) = e^(i*theta(t)) * L(1/2+it, chi)

q = 5
def theta_chi(t):
 return (t/2)*mp.log(q/mp.pi) + mp.arg(mp.gamma((mp.mpf('0.5') + 1j*t)/2))

def Z_chi(t):
 return mp.re(mp.exp(1j*theta_chi(t)) * mp.dirichlet(mp.mpf('0.5') + 1j*t, chi5))

# Test Z is real (imag close to 0)
for t in [1, 5, 6.5, 7, 12]:
 z = mp.exp(1j*theta_chi(t)) * mp.dirichlet(mp.mpf('0.5') + 1j*t, chi5)
 print(f"t={t}: Z={mp.re(z)}, im={mp.im(z)}")


t=1: Z=0.552589234618812735420241449424, im=-1.64049667899014250923843679609e-33
t=5: Z=2.13364394098992542112018940065, im=3.76544996900404058741323783598e-31
t=6.5: Z=0.24326001259148911848196765677, im=6.06824338284122424318877734848e-32
t=7: Z=-0.55869358601774049098932542445, im=-6.28304123812616648735140638258e-32
t=12: Z=-0.0936228612020484879312270767997, im=-6.49566414282966721039183750778e-32


In [10]:

# Perfect - Z is real. Now find zeros via sign changes.
# Known low zeros of L(s, chi) for chi_5 (Legendre mod 5) -- let's compute first ~30 zeros.
# We need this for LMFDB compatibility.

# Strategy: sign-change scan, then refine with mp.findroot at moderate dps.
# We need up to 20,000 zeros. Density of zeros for L(chi mod q) at height T is ~ (T/(2pi)) log(qT/(2pi)).
# Estimate height for 20000 zeros: N(T) ≈ (T/(2pi)) log(qT/(2pi))
# For q=5, solve T/(2pi) log(5T/(2pi)) = 20000 -> T ~ 20000 * 2pi / log(5*20000/(2pi)) 
# log(15915) ~ 9.67, so T ~ 20000*6.28/9.67 ~ 13000

# This will take time. Let's be more careful. Strategy: 
# - Use a sign-change scan with step ~0.1 to identify intervals, then refine with bisection/findroot.
# - dps=15 for the scan is enough (we then refine at dps=50, 80).

# First check feasibility on first 50 zeros.
mp.mp.dps = 25
import time

def find_sign_changes(t_start, t_end, step, dps=25):
 mp.mp.dps = dps
 ts = []
 zs = []
 t = mp.mpf(t_start)
 z_prev = Z_chi(t)
 while t < t_end:
 t_next = t + step
 z_next = Z_chi(t_next)
 if (z_prev > 0) != (z_next > 0):
 ts.append((t, t_next))
 z_prev = z_next
 t = t_next
 return ts

t0 = time.time()
intervals = find_sign_changes(0.5, 50, mp.mpf('0.1'), dps=20)
print(f"Found {len(intervals)} sign-changes up to t=50 in {time.time()-t0:.2f}s")
print("First 5:", intervals[:5])


Found 21 sign-changes up to t=50 in 3.31s
First 5: [(mpf('6.6000000000000000000556'), mpf('6.7000000000000000000569')), (mpf('9.7999999999999999999702'), mpf('9.8999999999999999999648')), (mpf('11.899999999999999999856'), mpf('11.999999999999999999851')), (mpf('15.999999999999999999634'), mpf('16.099999999999999999642')), (mpf('17.499999999999999999566'), mpf('17.599999999999999999561'))]


In [11]:

# Sign-change scan is slow (~3s for 50 t-values). For 20000 zeros at T~13000, with step 0.1 -> 130,000 evals.
# At ~6.6ms per eval = 14 minutes. Doable but a lot. 
# However, the research objective is primarily: 
# - download tables (Odlyzko zeta done; LMFDB blocked) 
# - validate R2 
# - REFINE to high precision
# - quantify speedup vs de novo
#
# For the L(chi) part, since LMFDB direct download fails, we must generate the "table" ourselves
# at moderate precision -- this IS de novo generation. So we should focus on showing speedup for ZETA
# (which is the larger computational concern) and document the LMFDB issue + generate the L(chi) zeros 
# de novo at limited count where feasible, OR use a more sophisticated approach.
#
# Better plan: try LMFDB via different routes (web sub-agent or direct API), and fall back to mpmath 
# generation if blocked.

# Try LMFDB with proper URL for the chi mod 5 L-function
import urllib.request

# LMFDB API for L-functions: try the new API endpoint
for url in [
 "https://www.lmfdb.org/L/data/Dirichlet/5/4",
 "https://www.lmfdb.org/L/Character/Dirichlet/5/4",
 "https://www.lmfdb.org/api/lfunctions/?conductor=5",
 "https://www.lmfdb.org/api/lfunctions",
 "https://beta.lmfdb.org/L/Character/Dirichlet/5/4",
]:
 try:
 req = urllib.request.Request(url, headers={'User-Agent':'Mozilla/5.0'})
 resp = urllib.request.urlopen(req, timeout=15)
 content = resp.read().decode('utf-8', errors='ignore')
 print(f"OK ({len(content)} bytes): {url}")
 # extract any "zeros" key
 if 'zero' in content.lower():
 i = content.lower().find('zero')
 print(" ...", content[max(0,i-100):i+500])
 break
 except Exception as e:
 print(f"FAIL: {url}: {e}")


FAIL: https://www.lmfdb.org/L/data/Dirichlet/5/4: HTTP Error 404: Not Found


OK (29100 bytes): https://www.lmfdb.org/L/Character/Dirichlet/5/4
 ... -r0-0-0">Euler factors to text</a></li>
 <li><a href="/L/download_zeros/1-5-5.4-r0-0-0">Zeros to text</a></li>
 <li><a href="/L/download_dirichlet_coeff/1-5-5.4-r0-0-0">Dirichlet coefficients to text</a></li>
 <li><a href="/L/data/1-5-5.4-r0-0-0">Underlying data</a></li>
 </ul>
 </div>
 
 <h1 class="properties-header">Learn more</h1>
 <div class="properties-body">
 <ul>
 <li><a href="/L/1/5/5.4/r0/0/0/Source">Source and acknow


In [12]:

# Found LMFDB endpoint for zeros! Try download_zeros for L-function 1-5-5.4-r0-0-0
url = "https://www.lmfdb.org/L/download_zeros/1-5-5.4-r0-0-0"
try:
 req = urllib.request.Request(url, headers={'User-Agent':'Mozilla/5.0'})
 resp = urllib.request.urlopen(req, timeout=30)
 content = resp.read().decode('utf-8', errors='ignore')
 print(f"Size: {len(content)}")
 print(content[:2000])
except Exception as e:
 print(f"FAIL: {e}")


Size: 9803

# Zeros of 1-5-5.4-r0-0-0 downloaded from the LMFDB on 30 May 2026.
{"order_of_vanishing": 0, "positive_zeros": ["6.64845334472771471612327845997", "9.831444432886669616348321347458", "11.95884562608351453026565868826", "16.03382112838423567459325378224", "17.566994292325555202701595268144", "19.54073262278475025037869002299", "22.227405454459410911877624963081", "24.58846621740819520765626997608", "26.77609594800414011652357496527", "28.46103510017752247518697827232", "29.70790935048096556923098651865", "33.00045600687051436794975917721", "34.728812978904808674143729833981", "35.86863837181227459459504863887", "38.12918472143653185015141827037", "39.56057294640318170505509505995", "41.84243854579169430850930688531", "44.03129006144169504470090805842", "45.4273000827822893888415619096", "46.49272715949140534533919935167", "48.34566182106784617654820130449", "51.08775192674649135525825720413", "52.1259022313169741886041306959", "53.83044519544216335105426902233", "55.5892803

In [13]:

# Excellent! LMFDB provides JSON with positive_zeros for L-function 1-5-5.4-r0-0-0.
# This is high-precision (~30 digits) already! Let me save and parse.
import json

with open("downloads/lmfdb_Lchi5.json", "w") as f:
 f.write(content)

data = json.loads(content)
print("Keys:", list(data.keys()))
print("Order of vanishing:", data['order_of_vanishing'])
zeros_str = data['positive_zeros']
print(f"Number of zeros: {len(zeros_str)}")
print(f"First 3: {zeros_str[:3]}")
print(f"Last 3: {zeros_str[-3:]}")


JSONDecodeError: Expecting value: line 2 column 1 (char 1)

In [14]:

# Skip the first comment line
lines = content.split('\n', 1)
print("Comment line:", lines[0])
data = json.loads(lines[1])
print("Keys:", list(data.keys()))
print("Order of vanishing:", data['order_of_vanishing'])
zeros_str = data['positive_zeros']
print(f"Number of zeros: {len(zeros_str)}")
print(f"First 5: {zeros_str[:5]}")
print(f"Last 3: {zeros_str[-3:]}")


Comment line: 


JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [15]:

# Find the JSON start
i = content.find('{')
data = json.loads(content[i:])
print("Keys:", list(data.keys()))
zeros_str = data['positive_zeros']
print(f"Number of zeros: {len(zeros_str)}")
print(f"First 5: {zeros_str[:5]}")
print(f"Last 3: {zeros_str[-3:]}")


Keys: ['order_of_vanishing', 'positive_zeros', 'negative_zeros', 'positive_zeros_accuracy', 'negative_zeros_accuracy']
Number of zeros: 129
First 5: ['6.64845334472771471612327845997', '9.831444432886669616348321347458', '11.95884562608351453026565868826', '16.03382112838423567459325378224', '17.566994292325555202701595268144']
Last 3: ['196.54076456244967267018733746529', '197.86100950772366771521398079814', '199.09102170624125905531999800239']


In [16]:

# LMFDB only has 129 zeros (up to t~199). That's far short of the 20000 needed.
# 
# Important finding: LMFDB does not provide tables of 20k zeros for Dirichlet L-functions.
# The "trusted source" approach (Odlyzko/LMFDB) is only feasible at scale for ζ; for L(χ) 
# the table is small. We need to either:
# (a) generate the remaining zeros de novo (sign-change scan, slower)
# (b) acknowledge the limitation and produce as many as feasible
# 
# Given compute budget, let me document this carefully. We'll:
# - For ζ: use Odlyzko (100k zeros) -> refine to 50, 80 dps for N in {2k,5k,10k,20k}
# - For L(χ): use LMFDB 129 zeros as the bootstrapping/validation set; for larger N, generate by 
# sign-change scan + refinement (which becomes the dominant cost). Show that for the 129 we have 
# fast refinement, and for the others we must do de novo work. Report extrapolated timings.

# First validate LMFDB R2-style for L(chi). The dataset description's R2 only specifies the 3 zeta zeros.
# Just sanity check first 3 zeros match our Z-function (sign changes from earlier)
mp.mp.dps = 30
lmfdb_Lchi5_zeros = [mp.mpf(s) for s in zeros_str]
print("First three LMFDB L(chi) zeros:")
for z in lmfdb_Lchi5_zeros[:3]:
 val = Z_chi(z)
 print(f" t={z}: Z(t) = {val}")


First three LMFDB L(chi) zeros:
 t=6.64845334472771471612327845997: Z(t) = 1.49613169688100519726713394929e-29
 t=9.83144443288666961634832134746: Z(t) = -1.79622789890435646015756288811e-30
 t=11.9588456260835145302656586883: Z(t) = 5.95602714397958410142215647966e-30


In [17]:

# All <1e-29 -- LMFDB zeros pass R2-like validation at full precision.
# 
# Now build the validation function and refinement function.

# First: ZETA part - this is the main story for the speedup.

# R2 gate validation function
def validate_R2_zeta(zeros, tol=mp.mpf('1e-6')):
 """Check first 3 zeros match R2 targets."""
 targets = [mp.mpf('14.134725141'), mp.mpf('21.022039639'), mp.mpf('25.010857580')]
 for i, (g, t) in enumerate(zip(zeros[:3], targets)):
 if abs(g - t) >= tol:
 return False, f"γ_{i+1}: {g} vs {t}, diff={abs(g-t)}"
 return True, "OK"

ok, msg = validate_R2_zeta(odlyzko_zeros)
print(f"Zeta R2 gate: pass={ok}, {msg}")

# Refinement function: use mpmath.findroot with Newton starting from low-precision seed
def refine_zeta_zero(t_seed, dps):
 mp.mp.dps = dps + 10 # extra guard digits
 rho = mp.findroot(lambda t: mp.zeta(mp.mpc(mp.mpf('0.5'), t)), mp.mpf(str(t_seed)),
 solver='newton')
 mp.mp.dps = dps
 return +rho # +rho rounds to current dps

# Time refinement of first zero at dps=80
import time
mp.mp.dps = 30
t_seed = odlyzko_zeros[0]
t0 = time.time()
rho = refine_zeta_zero(t_seed, 80)
dt = time.time() - t0
print(f"Refined γ_1 to dps=80: {rho}")
print(f"Time: {dt:.3f}s")
# Verify
mp.mp.dps = 80
print(f"|ζ(1/2+iγ_1)| = {abs(mp.zeta(mp.mpc(mp.mpf('0.5'), rho)))}")


Zeta R2 gate: pass=True, OK


TypeError: cannot create mpf from mpc(real='14.1347251417346937904630349069382769196529169931512793466223309990947560564270790222237926441711282', imag='-0.0000000000000000000285263387893088200783792511536660457072044352581898484228975066005192471094217652378894512238263483')

In [18]:

# Use siegelz (real-valued Hardy Z function) so the root variable is a real mpf.
# Z(t) = exp(i theta(t)) zeta(1/2 + i t) is real on the critical line.
# mpmath has mp.siegelz(t) which does exactly this. Find sign changes of siegelz.

def refine_zeta_zero(t_seed, dps):
 old = mp.mp.dps
 mp.mp.dps = dps + 15 # guard digits
 rho = mp.findroot(mp.siegelz, mp.mpf(t_seed), solver='newton')
 mp.mp.dps = dps
 rho_out = +rho
 mp.mp.dps = old
 return rho_out

mp.mp.dps = 30
import time
t_seed = odlyzko_zeros[0]
t0 = time.time()
rho = refine_zeta_zero(t_seed, 50)
dt50 = time.time() - t0
print(f"γ_1 @ dps=50: {rho}")
print(f" time: {dt50:.3f}s")

t0 = time.time()
rho = refine_zeta_zero(t_seed, 80)
dt80 = time.time() - t0
print(f"γ_1 @ dps=80: {rho}")
print(f" time: {dt80:.3f}s")

# Verify
mp.mp.dps = 90
print(f" |ζ(1/2+iρ)| = {abs(mp.zeta(mp.mpc(mp.mpf('0.5'), rho)))}")


γ_1 @ dps=50: 14.1347251417346937904572519836
 time: 0.092s
γ_1 @ dps=80: 14.1347251417346937904572519836
 time: 0.189s
 |ζ(1/2+iρ)| = 5.28444932798025999340407524672134949362396249206175171811855281505684394081070886173839288e-81


In [19]:

# Wait, dps=50 result is shown to only 30 digits because mp.mp.dps was reset to 30 by my final block. Let me reprint.
mp.mp.dps = 60
rho50 = refine_zeta_zero(odlyzko_zeros[0], 50)
mp.mp.dps = 60
print(f"γ_1 @ dps=50: {mp.nstr(rho50, 55)}")

mp.mp.dps = 90
rho80 = refine_zeta_zero(odlyzko_zeros[0], 80)
mp.mp.dps = 90
print(f"γ_1 @ dps=80: {mp.nstr(rho80, 85)}")

# Verify: |zeta(1/2 + i*rho80)|
mp.mp.dps = 90
val = abs(mp.zeta(mp.mpc(mp.mpf('0.5'), rho80)))
print(f" |ζ(1/2+iρ_80)| = {val}")


γ_1 @ dps=50: 14.13472514173469379045725198356247027078425711569924816
γ_1 @ dps=80: 14.13472514173469379045725198356247027078425711569924317568556746014996342980925675829
 |ζ(1/2+iρ_80)| = 5.28444932798025999340407524672134949362396249206175171811855281505684394081070886173839288e-81


In [20]:

# Excellent. The refinement works. Each zero takes ~0.1-0.2s.
# For N=20000 at dps=80: 20000 * 0.2s = 4000s = 67min. Significant but feasible.
# 
# But cost per zero grows with n (zeta evaluation at large height is more expensive).
# Let me time at a few different heights.

import time

mp.mp.dps = 30
test_indices = [1, 100, 1000, 5000, 10000, 20000]
print(f"{'n':>6} {'γ_n':>22} {'dps=50 (s)':>12} {'dps=80 (s)':>12}")
times_50 = {}
times_80 = {}
for n in test_indices:
 seed = odlyzko_zeros[n-1]
 t0 = time.time(); _ = refine_zeta_zero(seed, 50); t50 = time.time()-t0
 t0 = time.time(); _ = refine_zeta_zero(seed, 80); t80 = time.time()-t0
 times_50[n] = t50; times_80[n] = t80
 print(f"{n:>6} {float(seed):>22.6f} {t50:>12.4f} {t80:>12.4f}")


 n γ_n dps=50 (s) dps=80 (s)


 1 14.134725 0.0817 0.1879


 100 236.524230 0.1395 0.2995


 1000 1419.422481 1.1431 2.1767


 5000 5447.861998 2.3309 4.8416


 10000 9877.782654 3.5108 7.7265


 20000 18046.464296 5.7934 7.8800


In [21]:

# Cost per zero grows roughly like O(sqrt(t)) due to Riemann-Siegel formula cost.
# At n=20000: ~6s/zero at dps=50, ~8s at dps=80.
# Average for N=20000 will be perhaps ~3-4s/zero at dps=50 -> 60000-80000s = 17-22h. Too long!
# 
# For dps=80 with N=20000: extremely expensive.
# 
# We need to be more efficient. Options:
# 1. Reduce dps guard digits
# 2. Use the seed's accuracy: Odlyzko gives ~9 dp. Newton converges quadratically: 9 -> 18 -> 36 -> 72 -> 144.
# So 4-5 Newton iters get us to dps=80. But each Newton iter at dps=95 is expensive due to RS formula scaling.
# 3. Use lower dps for early refinement, then bump up.
# 
# Realistic approach: We must accept that for N=20000 at dps=80, full computation IS expensive (~hours).
# The hypothesis claim is "orders of magnitude faster than de novo". De novo means starting from scratch 
# with mp.zetazero(n), which also uses Riemann-Siegel but must FIND the zero (not just refine).
# Let me benchmark de novo to compare fairly.

# De novo timing at same indices:
print("De novo comparison (mp.zetazero):")
print(f"{'n':>6} {'γ_n (zetazero)':>22} {'time (s)':>12}")
denovo_times = {}
for n in [1, 100, 1000]:
 mp.mp.dps = 30
 t0 = time.time()
 z = mp.zetazero(n)
 dt = time.time() - t0
 denovo_times[n] = dt
 print(f"{n:>6} {float(mp.im(z)):>22.6f} {dt:>12.4f}")


De novo comparison (mp.zetazero):
 n γ_n (zetazero) time (s)
 1 14.134725 0.0206
 100 236.524230 0.1564


 1000 1419.422481 0.3257


In [22]:

# Interesting! mp.zetazero (de novo at dps=30) is FASTER than Newton from Odlyzko seed at dps=50.
# That's because dps=30 zetazero is much cheaper. 
# A fair comparison: at the TARGET dps. So:
# - DE NOVO at dps=50: mp.mp.dps=60; mp.zetazero(n)
# - REFINE at dps=50: starting from Odlyzko 9-digit seed
# 
# Let me benchmark fairly.

print("Fair comparison: de novo vs refine, both targeting dps=50 and dps=80")
print(f"{'n':>6} {'denovo50':>10} {'refine50':>10} {'denovo80':>10} {'refine80':>10}")

mp.mp.dps = 30 # reset

# De novo via zetazero
def denovo_zero(n, dps):
 old = mp.mp.dps
 mp.mp.dps = dps + 15
 z = mp.zetazero(n)
 mp.mp.dps = dps
 r = +mp.im(z)
 mp.mp.dps = old
 return r

denovo_50 = {}
denovo_80 = {}
refine_50 = {}
refine_80 = {}
for n in [1, 100, 1000]:
 seed = odlyzko_zeros[n-1]
 t0 = time.time(); _ = denovo_zero(n, 50); denovo_50[n] = time.time()-t0
 t0 = time.time(); _ = denovo_zero(n, 80); denovo_80[n] = time.time()-t0
 t0 = time.time(); _ = refine_zeta_zero(seed, 50); refine_50[n] = time.time()-t0
 t0 = time.time(); _ = refine_zeta_zero(seed, 80); refine_80[n] = time.time()-t0
 print(f"{n:>6} {denovo_50[n]:>10.3f} {refine_50[n]:>10.3f} {denovo_80[n]:>10.3f} {refine_80[n]:>10.3f}")


Fair comparison: de novo vs refine, both targeting dps=50 and dps=80
 n denovo50 refine50 denovo80 refine80


 1 0.024 0.075 0.026 0.174


 100 0.193 0.135 0.240 0.286


 1000 0.432 1.080 0.541 2.004


In [23]:

# CRUCIAL FINDING: mp.zetazero is actually FASTER than naive Newton refinement of Odlyzko seeds 
# at high precision.
#
# Why? mp.zetazero uses an efficient implementation internally. Naive `findroot(siegelz, seed, solver='newton')`
# at dps=95 evaluates siegelz repeatedly at high precision.
# 
# So the hypothesis "import-then-refine is orders of magnitude faster than de novo" is FALSE for zeta at high dps 
# when both methods use mpmath. mp.zetazero already implements an efficient algorithm.
# 
# However, refine_zeta_zero may be slow because:
# - We re-derive each derivative numerically. Let me try a smarter Newton: use mp.siegelz with derivative='dz'?
# - Or use bisection then Newton at target precision (only a few iters needed).
# 
# Let me try a tiered refinement: refine to dps=30 first, then to dps=80. This avoids many high-dps Newton iterations.

def refine_zeta_zero_tiered(t_seed, dps):
 """Refine via tiered precision doubling.
 Start at low dps, refine, increase, repeat."""
 # Start near double precision; Odlyzko gives ~9 digits
 current_dps = max(20, min(dps + 10, 30))
 old = mp.mp.dps
 mp.mp.dps = current_dps
 rho = mp.mpf(t_seed)
 # quick Newton at low dps
 rho = mp.findroot(mp.siegelz, rho, solver='newton', tol=mp.mpf('1e-25'))
 
 while current_dps < dps + 10:
 current_dps = min(current_dps * 2, dps + 15)
 mp.mp.dps = current_dps
 rho = mp.findroot(mp.siegelz, rho, solver='newton', maxsteps=10, 
 tol=mp.power(10, -(current_dps - 3)))
 
 mp.mp.dps = dps
 rho_out = +rho
 mp.mp.dps = old
 return rho_out

# Benchmark tiered
print("Tiered refinement:")
mp.mp.dps = 30
for n in [1, 100, 1000, 5000]:
 seed = odlyzko_zeros[n-1]
 t0 = time.time(); rho = refine_zeta_zero_tiered(seed, 50); t50 = time.time()-t0
 t0 = time.time(); rho = refine_zeta_zero_tiered(seed, 80); t80 = time.time()-t0
 print(f" n={n}: dps=50 in {t50:.3f}s, dps=80 in {t80:.3f}s")


Tiered refinement:
 n=1: dps=50 in 0.062s, dps=80 in 0.129s


 n=100: dps=50 in 0.201s, dps=80 in 0.315s


 n=1000: dps=50 in 0.995s, dps=80 in 1.914s


 n=5000: dps=50 in 2.534s, dps=80 in 4.692s


In [24]:

# Tiered is only marginally faster. The bottleneck is the final high-precision evaluation of siegelz.
# 
# Let me also benchmark with `findroot` using solver='secant' (avoids derivative computation):

def refine_secant(t_seed, dps):
 old = mp.mp.dps
 eps = mp.mpf('1e-5')
 mp.mp.dps = dps + 15
 rho = mp.findroot(mp.siegelz, [mp.mpf(t_seed) - eps, mp.mpf(t_seed) + eps], solver='anderson')
 mp.mp.dps = dps
 rho_out = +rho
 mp.mp.dps = old
 return rho_out

# Actually, let's just be pragmatic. The fairest test:
# 1. Refine N=2000 zeros at dps=50 from Odlyzko table 
# 2. Compare to de-novo time extrapolated from N=100 sample

# Let me commit to: refine N in {2000, 5000, 10000, 20000} at dps=50 and dps=80, and ESTIMATE
# de novo time from a smaller sample. The full N=20000 at dps=80 is computationally prohibitive 
# (~4 hours), so we'll fully execute the smaller grids and extrapolate for the largest.

# Practical plan:
# - For each (N, dps): time refinement of FIRST 50 zeros, FIRST 200 zeros, plus a small sample at higher indices
# - Estimate full time by integrating cost(n) over n from 1..N

# Even better: we can write the script as the spec demands, run it on N=2000 fully, and provide 
# extrapolations for the rest along with the speedup quantification.

# Let me just RUN N=2000 dps=50 in full and N=2000 dps=80 in full, plus sample-based extrapolations for others.

# Time budget: total runtime budget is 3600s. We've used some. Let's allocate:
# - N=2000 dps=50: ~ 2000 * 1.5 avg = ~3000s ... too long for largest N
# 
# Reconsider: compute average time per zero from samples and extrapolate; only fully compute 
# the smaller grids. Let me look at: 
# - For Odlyzko first 2000 zeros: max index 2000, γ ~ 2500
# - Avg refinement time at dps=50 over n=1..2000 estimated:

# Fit cost(n) ~ a + b * n^0.5 (heuristic)
import numpy as np
ns = np.array([1, 100, 1000, 5000])
ts50 = np.array([0.075, 0.135, 1.080, 2.331])
ts80 = np.array([0.174, 0.286, 2.004, 4.842])
# Fit power law t = a*n^b 
# Using log-log linear regression on (n, t)
for ts, label in [(ts50, 'dps=50'), (ts80, 'dps=80')]:
 p = np.polyfit(np.log(ns), np.log(ts), 1)
 print(f"{label}: t ~ {np.exp(p[1]):.4f} * n^{p[0]:.3f}")


dps=50: t ~ 0.0512 * n^0.410
dps=80: t ~ 0.1174 * n^0.392


In [25]:

# OK so cost(n) ~ a * n^0.4 roughly.
# Integral_0^N of a*n^0.4 dn = a * N^1.4 / 1.4
# For dps=50, a~0.05: total(N=20000) = 0.05 * 20000^1.4 / 1.4 ~ 0.05 * 851138/1.4 ~ 30,400s = 8.4h
# For dps=80, a~0.12: total(N=20000) = 0.12 * 20000^1.4 / 1.4 ~ 73,000s = 20h
# 
# So even N=20000 with refinement is prohibitive.
# Compare with de novo. For mp.zetazero:
# de novo at dps=50: at n=1000 is 0.4s; scaling as n^0.4 likely.
# Let me sample mp.zetazero at higher n.

import time
print("De-novo (mp.zetazero) at various n:")
print(f"{'n':>6} {'dps=50 (s)':>12} {'dps=80 (s)':>12}")
denovo_data = []
for n in [1, 100, 1000, 5000]:
 t0 = time.time(); _ = denovo_zero(n, 50); t50 = time.time()-t0
 t0 = time.time(); _ = denovo_zero(n, 80); t80 = time.time()-t0
 denovo_data.append((n, t50, t80))
 print(f"{n:>6} {t50:>12.4f} {t80:>12.4f}")


De-novo (mp.zetazero) at various n:
 n dps=50 (s) dps=80 (s)
 1 0.0234 0.0269


 100 0.1979 0.2385


 1000 0.4304 0.5600


 5000 0.8217 1.3428


In [26]:

# CRITICAL FINDING confirmed:
# mp.zetazero is CONSISTENTLY FASTER than naive Newton refinement at the same target dps.
# - At n=5000, dps=80: zetazero takes 1.34s; refinement takes ~4.8s (~3.6x SLOWER).
# - At n=1000, dps=80: zetazero takes 0.56s; refinement 2.0s.
# 
# Therefore: The research hypothesis "Importing pre-computed zero tables and refining them with mpmath 
# will be orders of magnitude faster than de novo" is FALSE for the Riemann zeta function when 
# "de novo" means mp.zetazero (which uses Riemann-Siegel internally and is highly optimized in mpmath).
# 
# The hypothesis would be true if de novo meant something naive like a sign-change scan from scratch.
# But mpmath's zetazero IS efficient.
#
# Now produce the cached files as the spec requires, but use de novo (mp.zetazero) since it's faster,
# AND ALSO save Odlyzko-refined versions for the smaller N to demonstrate the comparison.
# Actually, the spec says: refine the imported zeros and save them.
# So we follow the spec, save the refined zeros, but ALSO honestly report that refinement was 
# slower than de novo for ζ.
# 
# Time budget: we have 3600s. Already used some. Let me budget:
# - ζ N=2000 dps=50: ~2000 * 0.3 = 600s (refinement) -- too long 
# - ζ N=2000 dps=80: ~2000 * 0.6 = 1200s
# 
# Total budget consumption if we do all refinements: prohibitive.
# 
# Realistic plan:
# - Fully refine the first N=2000 at dps=50 and dps=80 (demonstrating the protocol)
# - For larger N (5000, 10000, 20000), provide EXTRAPOLATED estimates and partial samples
# - The PRIMARY cached file is N=2000_dps50, N=2000_dps80
# - Save the FULL Odlyzko table itself as a low-precision cache
# - Document the speedup analysis quantitatively
# 
# Actually wait - rethink. Maybe I'm being inefficient with findroot. Let me try one more approach:
# Use mp.findroot with just 1-2 Newton steps at high dps starting from a *medium* dps refinement.

def refine_zeta_zero_smart(t_seed, dps):
 """
 Smart refinement: 
 1) Use Odlyzko seed (~9 dp) at dps=20 to refine -> ~20 dp answer
 2) Use that to refine at target dps with just 2-3 Newton steps
 """
 old = mp.mp.dps
 # Step 1: dps=25 quick refine
 mp.mp.dps = 25
 rho = mp.findroot(mp.siegelz, mp.mpf(t_seed), solver='newton', tol=mp.mpf('1e-20'))
 # Step 2: at target dps, just a couple Newton steps (Newton converges quadratically: 20 -> 40 -> 80)
 mp.mp.dps = dps + 10
 rho = mp.findroot(mp.siegelz, rho, solver='newton', maxsteps=5, tol=mp.power(10, -(dps+5)))
 mp.mp.dps = dps
 rho_out = +rho
 mp.mp.dps = old
 return rho_out

print("Smart refinement timing:")
for n in [1, 100, 1000, 5000]:
 seed = odlyzko_zeros[n-1]
 t0 = time.time(); r50 = refine_zeta_zero_smart(seed, 50); t50 = time.time()-t0
 t0 = time.time(); r80 = refine_zeta_zero_smart(seed, 80); t80 = time.time()-t0
 print(f" n={n}: dps=50 in {t50:.4f}s, dps=80 in {t80:.4f}s")

# Verify
mp.mp.dps = 90
v = abs(mp.zeta(mp.mpc(mp.mpf('0.5'), r80)))
print(f" validation |ζ(1/2+iρ)| at n=5000 dps=80: {v}")


Smart refinement timing:
 n=1: dps=50 in 0.0536s, dps=80 in 0.1129s


 n=100: dps=50 in 0.1604s, dps=80 in 0.2493s


 n=1000: dps=50 in 0.9262s, dps=80 in 1.6158s


 n=5000: dps=50 in 1.8785s, dps=80 in 3.5589s
 validation |ζ(1/2+iρ)| at n=5000 dps=80: 1.66422932861723250544739086623419090017089445607169117846375786048563367996128686186169344e-77


In [27]:

# Smart refinement is ~25-30% faster than naive but still slower than de novo.
# Now I have a clear story. Let me proceed with the cached file generation.
#
# DECISION: 
# - I will use REFINEMENT (per the spec) for the cached zero lists.
# - I will fully execute N=2000 dps=50 and dps=80 to produce real cached files.
# - For N=5000, 10000, 20000, I will provide computational estimates and partial samples.
# - The primary scientific finding will be that for ζ, mpmath's de novo `zetazero` is actually 
# FASTER than Newton refinement of Odlyzko seeds — the hypothesis is refuted.
# - For L(χ_4 mod 5), LMFDB only provides ~129 zeros, far short of 20k. So the "import-then-refine" 
# strategy is severely capacity-limited for L-functions. We can refine all 129 to dps=50/80.

# Let me start: produce ζ cached file at N=2000, dps=50. Track total time.

import time

# Save Odlyzko raw zeros table (low-precision cache)
with open('cached_zeros/zeta/odlyzko_zeros1_raw.txt', 'w') as f:
 f.write("# Odlyzko zeros1: first 100,000 imaginary parts of nontrivial ζ zeros (~9-10 dp)\n")
 f.write(f"# Source: https://www-users.cse.umn.edu/~odlyzko/zeta_tables/zeros1\n")
 f.write(f"# Downloaded: 2026-05-30\n")
 for z in odlyzko_zeros:
 f.write(f"{z}\n")
print(f"Saved low-precision Odlyzko zeta zeros: cached_zeros/zeta/odlyzko_zeros1_raw.txt")
print(f" {len(odlyzko_zeros)} zeros")


Saved low-precision Odlyzko zeta zeros: cached_zeros/zeta/odlyzko_zeros1_raw.txt
 100000 zeros


In [28]:

# Now let me allocate the budget. We have plenty of time left. 
# Estimated full runs (refinement) for each (N, dps):
# Using smart refinement, cost ~ k*n^0.4 where k depends on dps.
# Total time for N zeros = sum_{n=1}^N k*n^0.4 ~ k * N^1.4 / 1.4
# dps=50: k ~ 0.043 (from n=5000, t=1.88 -> 1.88/5000^0.4 = 0.043)
# dps=80: k ~ 0.083 (3.56/5000^0.4 = 0.083)
# 
# Total estimated time:
import numpy as np
k50 = 1.88 / (5000**0.4)
k80 = 3.56 / (5000**0.4)
for N in [2000, 5000, 10000, 20000]:
 t50_total = k50 * (N**1.4) / 1.4
 t80_total = k80 * (N**1.4) / 1.4
 print(f" N={N}: estimated dps=50 total: {t50_total/60:.1f} min, dps=80 total: {t80_total/60:.1f} min")


 N=2000: estimated dps=50 total: 31.0 min, dps=80 total: 58.8 min
 N=5000: estimated dps=50 total: 111.9 min, dps=80 total: 211.9 min
 N=10000: estimated dps=50 total: 295.3 min, dps=80 total: 559.2 min
 N=20000: estimated dps=50 total: 779.4 min, dps=80 total: 1475.8 min


In [29]:

# Even N=2000 dps=50 will take ~30 min. With N=2000 dps=80 ~60 min, we'd use 90 min for those two alone.
# Total budget remaining: ~50 min after overhead. We can't afford even N=2000 fully.
# 
# REVISED PLAN: 
# - Refine ONLY a sample (e.g., first 200 zeros and last 100 zeros of N=2000) at both dps, 
# to demonstrate correctness and timing scaling
# - For each (N, dps), provide:
# * Refined zeros for the first 200 (cached as "_partial.txt") 
# * Detailed timing extrapolation
# - For LMFDB L(chi5), refine all 129 zeros at both dps (this IS small and fast)
# 
# This honestly demonstrates the protocol while staying within compute budget.
# AND it produces the key quantitative comparison: refinement vs. de novo per zero.

# Even better: I can fully refine N=2000 at dps=50 ONLY (~30 min) and dps=80 for first 500 zeros (~5-10 min).
# Then extrapolate for larger N.

# Let me check current time elapsed approx
print("Budget plan:")
print(" - ζ N=2000 dps=50: full refine (~30 min) -- main demonstration")
print(" - ζ N=2000 dps=80: refine first 200 (~3 min) + extrapolation")
print(" - For N=5k, 10k, 20k: extrapolation only")
print(" - L(χ) all 129 zeros at dps=50, dps=80: full refine (~few min)")

# Actually let's be more conservative. Do FULL refine for N=2000 only at dps=50.
# Skip dps=80 full but sample 100 zeros at dps=80.
# Total estimated time: ~30 min + 5 min = 35 min, leaves margin for L(χ) work and writeup.

# Actually I realize Newton from a good seed may converge in just 1-2 iters. Let me check more carefully.
# The bottleneck per zero is the cost of evaluating siegelz at high dps, which scales as O(sqrt(t)) 
# via Riemann-Siegel.

# Let me try with much fewer Newton steps explicitly (since seed is already 9 digits, 
# 1 Newton step at dps=20 gives 18 digits, 1 at dps=40 gives 36 digits, 1 at dps=90 gives ~72 digits 
# which is barely insufficient for dps=80 -- need 2 steps)

def refine_explicit(t_seed, dps):
 """Explicit Newton iterations with precision doubling."""
 old = mp.mp.dps
 rho = mp.mpf(t_seed) # ~9 dp
 # Iteration 1: dps=20 -> get ~18 dp
 mp.mp.dps = 22
 rho = rho - mp.siegelz(rho) / mp.siegelz(rho, derivative=1)
 # Iteration 2: dps=40 -> get ~36 dp
 if dps > 18:
 mp.mp.dps = 45
 rho = rho - mp.siegelz(rho) / mp.siegelz(rho, derivative=1)
 # Iteration 3: dps=80 -> get ~72 dp
 if dps > 36:
 mp.mp.dps = max(85, dps+10)
 rho = rho - mp.siegelz(rho) / mp.siegelz(rho, derivative=1)
 # Iteration 4 at full dps if needed
 if dps > 70:
 mp.mp.dps = dps + 15
 rho = rho - mp.siegelz(rho) / mp.siegelz(rho, derivative=1)
 mp.mp.dps = dps
 out = +rho
 mp.mp.dps = old
 return out

print("Explicit Newton timing:")
for n in [1, 100, 1000, 5000]:
 seed = odlyzko_zeros[n-1]
 t0 = time.time(); r50 = refine_explicit(seed, 50); t50 = time.time()-t0
 t0 = time.time(); r80 = refine_explicit(seed, 80); t80 = time.time()-t0
 # validate
 mp.mp.dps = 90
 val = abs(mp.zeta(mp.mpc(mp.mpf('0.5'), r80)))
 print(f" n={n}: dps=50 in {t50:.4f}s, dps=80 in {t80:.4f}s, |ζ| at dps80 result: {float(val):.2e}")


Budget plan:
 - ζ N=2000 dps=50: full refine (~30 min) -- main demonstration
 - ζ N=2000 dps=80: refine first 200 (~3 min) + extrapolation
 - For N=5k, 10k, 20k: extrapolation only
 - L(χ) all 129 zeros at dps=50, dps=80: full refine (~few min)
Explicit Newton timing:
 n=1: dps=50 in 0.0327s, dps=80 in 0.0548s, |ζ| at dps80 result: 5.28e-81


 n=100: dps=50 in 0.1147s, dps=80 in 0.1543s, |ζ| at dps80 result: 3.50e-80


 n=1000: dps=50 in 0.3794s, dps=80 in 0.6029s, |ζ| at dps80 result: 2.92e-78


 n=5000: dps=50 in 0.8131s, dps=80 in 1.2832s, |ζ| at dps80 result: 1.66e-77


In [30]:

# MUCH BETTER! Explicit Newton with precision doubling:
# - n=5000 dps=80: 1.28s (vs 4.84s with naive findroot, ~3.8x faster)
# - n=5000 dps=50: 0.81s
# 
# And the result is verified: |ζ| < 1e-77 at dps=80, which means the zero is accurate to ~77 digits.
# 
# Re-estimate total times:
k50 = 0.813 / (5000**0.4)
k80 = 1.283 / (5000**0.4)
for N in [2000, 5000, 10000, 20000]:
 t50_total = k50 * (N**1.4) / 1.4
 t80_total = k80 * (N**1.4) / 1.4
 print(f" N={N}: dps=50 ~ {t50_total/60:.1f} min, dps=80 ~ {t80_total/60:.1f} min")


 N=2000: dps=50 ~ 13.4 min, dps=80 ~ 21.2 min
 N=5000: dps=50 ~ 48.4 min, dps=80 ~ 76.4 min
 N=10000: dps=50 ~ 127.7 min, dps=80 ~ 201.5 min
 N=20000: dps=50 ~ 337.0 min, dps=80 ~ 531.9 min


In [31]:

# Better but still: N=2000 alone uses ~35min of our budget. 
# Compare to de novo (mp.zetazero):
k50_dn = 0.821 / (5000**0.4) # from de novo at n=5000, t=0.82s
k80_dn = 1.343 / (5000**0.4)
for N in [2000, 5000, 10000, 20000]:
 t50_total = k50_dn * (N**1.4) / 1.4
 t80_total = k80_dn * (N**1.4) / 1.4
 print(f" De novo N={N}: dps=50 ~ {t50_total/60:.1f} min, dps=80 ~ {t80_total/60:.1f} min")

# So refinement (explicit Newton from Odlyzko seed) is NOW comparable to or slightly better than 
# de novo (mp.zetazero). NOT orders of magnitude faster.
# 
# Key fact: mp.zetazero internally also uses Riemann-Siegel + Newton; it just doesn't have a 
# precomputed seed. The Odlyzko table provides a 9-digit seed, saving the "search" step. But the 
# refinement cost at high dps dominates.
# 
# For the hypothesis ("orders of magnitude faster"), the answer is: NO. The speedup is at best 
# a factor of ~1-2x, not orders of magnitude.

# OK let me now produce the cached files. I'll do:
# - N=2000 dps=50 fully via refinement (13 min) -- main artifact
# - First 200 zeros at all (N, dps) combos as samples + extrapolations
# - L(χ) all 129 zeros at dps=50 and dps=80 fully
# 
# Let's start the N=2000 dps=50 full run.

import time

def refine_batch(seeds, dps, verbose_every=200):
 """Refine a batch of zeros at target dps using explicit Newton."""
 refined = []
 t_start = time.time()
 for i, s in enumerate(seeds):
 refined.append(refine_explicit(s, dps))
 if (i+1) % verbose_every == 0:
 elapsed = time.time() - t_start
 rate = (i+1) / elapsed
 eta = (len(seeds) - i - 1) / rate
 print(f" {i+1}/{len(seeds)}, elapsed {elapsed:.1f}s, rate {rate:.1f}/s, ETA {eta:.1f}s")
 return refined, time.time() - t_start

# Run N=2000 dps=50
N = 2000
target_dps = 50
print(f"Refining first N={N} zeta zeros to dps={target_dps}...")
seeds = odlyzko_zeros[:N]
refined_2000_50, time_2000_50 = refine_batch(seeds, target_dps, verbose_every=200)
print(f"DONE: {N} zeros at dps={target_dps} in {time_2000_50:.1f}s = {time_2000_50/60:.2f}min")


 De novo N=2000: dps=50 ~ 13.5 min, dps=80 ~ 22.2 min
 De novo N=5000: dps=50 ~ 48.9 min, dps=80 ~ 79.9 min
 De novo N=10000: dps=50 ~ 129.0 min, dps=80 ~ 211.0 min
 De novo N=20000: dps=50 ~ 340.3 min, dps=80 ~ 556.7 min
Refining first N=2000 zeta zeros to dps=50...


 200/2000, elapsed 21.7s, rate 9.2/s, ETA 195.2s


 400/2000, elapsed 68.2s, rate 5.9/s, ETA 272.8s


 600/2000, elapsed 123.9s, rate 4.8/s, ETA 289.2s


 800/2000, elapsed 190.3s, rate 4.2/s, ETA 285.4s


...

 1200/2000, elapsed 340.7s, rate 3.5/s, ETA 227.2s


 1400/2000, elapsed 418.9s, rate 3.3/s, ETA 179.5s


 1600/2000, elapsed 506.2s, rate 3.2/s, ETA 126.5s


 1800/2000, elapsed 599.7s, rate 3.0/s, ETA 66.6s


 2000/2000, elapsed 700.8s, rate 2.9/s, ETA 0.0s
DONE: 2000 zeros at dps=50 in 700.8s = 11.68min


In [32]:

# Done in 11.7 min for N=2000 dps=50. Now save the cache.

cache_path = 'cached_zeros/zeta/N2000_dps50.txt'
with open(cache_path, 'w') as f:
 f.write(f"# Refined imaginary parts of first {N} non-trivial zeros of ζ(s)\n")
 f.write(f"# Source: Odlyzko zeros1 seeded, Newton-refined via mpmath.siegelz\n")
 f.write(f"# Working precision: dps={target_dps}\n")
 f.write(f"# Refinement time: {time_2000_50:.1f}s\n")
 for z in refined_2000_50:
 f.write(mp.nstr(z, target_dps + 2, strip_zeros=False) + "\n")
print(f"Saved: {cache_path}")

# Validate R2 on the refined zeros
mp.mp.dps = 60
print("R2 validation on refined zeros:")
print(f" γ_1 = {mp.nstr(refined_2000_50[0], 30)}")
print(f" γ_2 = {mp.nstr(refined_2000_50[1], 30)}")
print(f" γ_3 = {mp.nstr(refined_2000_50[2], 30)}")
# Verify |ζ(1/2+iρ)| is tiny
for i, g in enumerate(refined_2000_50[:3]):
 v = abs(mp.zeta(mp.mpc(mp.mpf('0.5'), g)))
 print(f" |ζ(1/2 + iγ_{i+1})| = {float(v):.3e}")


Saved: cached_zeros/zeta/N2000_dps50.txt
R2 validation on refined zeros:
 γ_1 = 14.1347251417346937904572519836
 γ_2 = 21.0220396387715549926284795939
 γ_3 = 25.0108575801456887632137909926
 |ζ(1/2 + iγ_1)| = 3.954e-51
 |ζ(1/2 + iγ_2)| = 2.078e-51
 |ζ(1/2 + iγ_3)| = 2.368e-50


In [33]:

# R2 passes. |ζ| ~ 10^-50 confirms dps=50 precision.
# 
# Now refine same N=2000 at dps=80 for a SAMPLE (first 200 + last 200) and extrapolate full.
# Actually let me just do the first 500 at dps=80 for a representative timing.

# Budget remaining: I've used ~11 min on the main run plus benchmarks. Let me do a 5 min budget here.
# refining ~500 zeros at dps=80 should take ~3-5 min based on prior measurements.

N_sample = 500
print(f"Refining first {N_sample} zeta zeros to dps=80...")
seeds_500 = odlyzko_zeros[:N_sample]
refined_500_80, time_500_80 = refine_batch(seeds_500, 80, verbose_every=100)
print(f"DONE: {N_sample} zeros at dps=80 in {time_500_80:.1f}s ({time_500_80/60:.2f}min)")
print(f"Avg time per zero: {time_500_80/N_sample*1000:.1f} ms")


Refining first 500 zeta zeros to dps=80...


 100/500, elapsed 8.7s, rate 11.5/s, ETA 34.9s


 200/500, elapsed 29.3s, rate 6.8/s, ETA 43.9s


 300/500, elapsed 60.5s, rate 5.0/s, ETA 40.3s


 400/500, elapsed 100.8s, rate 4.0/s, ETA 25.2s


 500/500, elapsed 143.1s, rate 3.5/s, ETA 0.0s
DONE: 500 zeros at dps=80 in 143.1s (2.38min)
Avg time per zero: 286.1 ms


In [34]:

# Save partial cache at dps=80
with open('cached_zeros/zeta/N2000_dps80_partial500.txt', 'w') as f:
 f.write(f"# Refined ζ zeros (PARTIAL: first 500 of 2000) at dps=80\n")
 f.write(f"# Source: Odlyzko seeded, Newton-refined\n")
 f.write(f"# Time: {time_500_80:.1f}s for 500 zeros\n")
 for z in refined_500_80:
 f.write(mp.nstr(z, 82, strip_zeros=False) + "\n")
print("Saved partial dps=80 cache")

# Now let me also produce dps=80 cache for the FULL N=2000. Estimate: time scaling implies
# total ~21 min. I have time but it's a lot.
# 
# Actually let me also check: can the LARGER N caches be built from de novo zetazero faster?
# This would let me document the actual best-method timing.
# 
# But the spec says: "Use a root-finding method in mpmath (e.g., Newton's method via findroot) to refine"
# So refinement is the spec.
# 
# Let me proceed pragmatically. Skip full dps=80 for now (will extrapolate from 500 sample).
# Move to L(chi) refinement (small task).

# L(chi) refinement - all 129 LMFDB zeros at dps=50 and dps=80
print("Refining L(s, chi_4 mod 5) zeros from LMFDB seeds...")

# Refinement function for L(chi):
def refine_Lchi(t_seed, dps):
 """Refine an L(s, chi_4 mod 5) zero on the critical line using explicit Newton on Z_chi."""
 old = mp.mp.dps
 rho = mp.mpf(t_seed)
 
 # Define Z and its derivative explicitly via numeric differentiation
 def Z(t, p):
 mp.mp.dps = p
 return mp.re(mp.exp(1j * theta_chi(t)) * mp.dirichlet(mp.mpf('0.5') + 1j*t, chi5))
 
 # Iterative Newton with precision doubling. Use simple finite differences for derivative? 
 # For L-functions, derivative of dirichlet is supported.
 
 # mp.dirichlet has a derivative parameter. d/ds dirichlet(s, chi) = -sum log(k) chi(k)/k^s
 # We need dZ/dt = d/dt of real part of exp(i*theta)*L(1/2+it)
 # Easier: numerical derivative.
 
 def Zprime(t, p, h=None):
 if h is None:
 h = mp.power(10, -(p // 3))
 return (Z(t + h, p) - Z(t - h, p)) / (2 * h)
 
 # Newton iterations
 targets = [22, 45, 85] if dps <= 70 else [22, 45, 85, dps+15]
 targets = [p for p in targets if p <= dps + 15] + ([dps + 15] if dps + 15 > max([22, 45, 85][:min(3, len(targets))]) else [])
 targets = sorted(set(targets))
 
 for p in targets:
 mp.mp.dps = p
 rho = rho - Z(rho, p) / Zprime(rho, p)
 
 mp.mp.dps = dps
 out = +rho
 mp.mp.dps = old
 return out

# Test on first zero
mp.mp.dps = 30
seed = mp.mpf(zeros_str[0])
t0 = time.time()
r50 = refine_Lchi(seed, 50)
t50 = time.time() - t0
print(f" L(chi) zero #1 at dps=50: {r50}")
print(f" time: {t50:.3f}s")
# Validate
mp.mp.dps = 60
v = abs(mp.dirichlet(mp.mpf('0.5') + 1j*r50, chi5))
print(f" |L(1/2+iρ, chi)| = {float(v):.3e}")


Saved partial dps=80 cache
Refining L(s, chi_4 mod 5) zeros from LMFDB seeds...
 L(chi) zero #1 at dps=50: 6.64845334472771471612327845998
 time: 0.053s
 |L(1/2+iρ, chi)| = 4.998e-46


In [35]:

# L(chi) refinement is fast and accurate.
# Refine all 129 LMFDB zeros at dps=50 and dps=80.

t0 = time.time()
lchi_refined_50 = [refine_Lchi(mp.mpf(s), 50) for s in zeros_str]
time_lchi_50 = time.time() - t0
print(f"L(chi): 129 zeros at dps=50 in {time_lchi_50:.2f}s")

t0 = time.time()
lchi_refined_80 = [refine_Lchi(mp.mpf(s), 80) for s in zeros_str]
time_lchi_80 = time.time() - t0
print(f"L(chi): 129 zeros at dps=80 in {time_lchi_80:.2f}s")

# Validate by checking |L(1/2 + i*rho)| is tiny
mp.mp.dps = 90
worst_50 = max(abs(mp.dirichlet(mp.mpf('0.5') + 1j*r, chi5)) for r in lchi_refined_50)
worst_80 = max(abs(mp.dirichlet(mp.mpf('0.5') + 1j*r, chi5)) for r in lchi_refined_80)
print(f"\nValidation: max |L(1/2+iρ, chi)| at dps=50: {float(worst_50):.3e}")
print(f"Validation: max |L(1/2+iρ, chi)| at dps=80: {float(worst_80):.3e}")


L(chi): 129 zeros at dps=50 in 16.73s


L(chi): 129 zeros at dps=80 in 44.14s



Validation: max |L(1/2+iρ, chi)| at dps=50: 3.367e-41
Validation: max |L(1/2+iρ, chi)| at dps=80: 1.181e-78


In [36]:

# Save L(chi) caches
for arr, dps in [(lchi_refined_50, 50), (lchi_refined_80, 80)]:
 path = f'cached_zeros/Lchi5/N129_dps{dps}.txt'
 with open(path, 'w') as f:
 f.write(f"# Refined imaginary parts of zeros of L(s, χ_4 mod 5), character Dirichlet 5.4 (LMFDB label 1-5-5.4-r0-0-0)\n")
 f.write(f"# Source: LMFDB download_zeros (positive_zeros, ~30 dp), Newton-refined via Hardy Z\n")
 f.write(f"# Note: LMFDB provides only {len(arr)} zeros (up to t≈199)\n")
 f.write(f"# Working precision: dps={dps}\n")
 for z in arr:
 f.write(mp.nstr(z, dps + 2, strip_zeros=False) + "\n")
 print(f"Saved: {path}")


Saved: cached_zeros/Lchi5/N129_dps50.txt
Saved: cached_zeros/Lchi5/N129_dps80.txt


In [37]:

# Now let me also try to fully refine N=2000 at dps=80 (estimated ~20 min). I should have budget.
# Actually I've used roughly: setup ~5min + 11.7min N2000_dps50 + 2.4min sample + 1min L(chi) = ~20min.
# Remaining ~40 min. Let me do N=2000 dps=80.

# Resume from already refined 500 zeros and add the rest
import time

print("Refining ζ zeros 501-2000 at dps=80 (continuing)...")
seeds_remaining = odlyzko_zeros[500:2000]
refined_remaining_80, time_remaining_80 = refine_batch(seeds_remaining, 80, verbose_every=200)
print(f"DONE: {len(seeds_remaining)} more zeros in {time_remaining_80:.1f}s")

all_refined_2000_80 = refined_500_80 + refined_remaining_80
total_time_2000_80 = time_500_80 + time_remaining_80
print(f"Total: {len(all_refined_2000_80)} zeros at dps=80 in {total_time_2000_80:.1f}s = {total_time_2000_80/60:.2f}min")


TimeoutError: Code execution timed out after 900.0 seconds

In [38]:

# Cell timed out at 900s. Need to check what got done.
print(f"len refined_remaining_80: {len(refined_remaining_80) if 'refined_remaining_80' in dir() else 'undefined'}")


len refined_remaining_80: undefined


In [39]:

# Cell timed out; partial work lost. We have refined_500_80 from cached_zeros file though.
# Let's just save what we have and move on. We can do additional batches but be careful.

# Restart - just resume from disk
# Already saved: cached_zeros/zeta/N2000_dps50.txt (2000 zeros at dps=50)
# cached_zeros/zeta/N2000_dps80_partial500.txt (500 zeros at dps=80)
# cached_zeros/Lchi5/N129_dps50.txt
# cached_zeros/Lchi5/N129_dps80.txt

# Variables refined_2000_50 and lchi_refined* should still be in scope. Let's check.
print(f"refined_2000_50 length: {len(refined_2000_50) if 'refined_2000_50' in dir() else 'NA'}")
print(f"refined_500_80 length: {len(refined_500_80) if 'refined_500_80' in dir() else 'NA'}")
print(f"lchi_refined_50 length: {len(lchi_refined_50) if 'lchi_refined_50' in dir() else 'NA'}")
print(f"lchi_refined_80 length: {len(lchi_refined_80) if 'lchi_refined_80' in dir() else 'NA'}")
print(f"time_2000_50: {time_2000_50}")
print(f"time_500_80: {time_500_80}")


refined_2000_50 length: 2000
refined_500_80 length: 500
lchi_refined_50 length: 129
lchi_refined_80 length: 129
time_2000_50: 700.8419897556305
time_500_80: 143.06847143173218


In [40]:

# Variables preserved! Now let me do a small additional batch at dps=80 within the cell timeout.
# Do 200 more (zeros 501-700) at dps=80 ~ should take 1-2 min.

import time
print("Refining zeros 501-700 at dps=80 (batch 2 of 7)...")
seeds_701_900 = odlyzko_zeros[500:700]
t0 = time.time()
batch_701_900 = []
for i, s in enumerate(seeds_701_900):
 batch_701_900.append(refine_explicit(s, 80))
 if (i+1) % 50 == 0:
 print(f" {i+1}/200, elapsed {time.time()-t0:.1f}s")
time_701_900 = time.time() - t0
print(f"DONE: 200 zeros in {time_701_900:.1f}s")

# Save partial: now first 700 dps=80
all_700_80 = refined_500_80 + batch_701_900
with open('cached_zeros/zeta/N2000_dps80_partial700.txt', 'w') as f:
 f.write(f"# Refined ζ zeros (PARTIAL: first 700 of 2000) at dps=80\n")
 f.write(f"# Source: Odlyzko seeded, Newton-refined\n")
 f.write(f"# Time: {time_500_80 + time_701_900:.1f}s for 700 zeros\n")
 for z in all_700_80:
 f.write(mp.nstr(z, 82, strip_zeros=False) + "\n")
print("Saved cached_zeros/zeta/N2000_dps80_partial700.txt")


Refining zeros 501-700 at dps=80 (batch 2 of 7)...


 50/200, elapsed 21.5s


 100/200, elapsed 43.2s


 150/200, elapsed 65.3s


 200/200, elapsed 89.4s
DONE: 200 zeros in 89.4s
Saved cached_zeros/zeta/N2000_dps80_partial700.txt


In [41]:

# OK time to wrap up. Let me compute the speedup analysis and finalize.
# 
# Key timings collected:
# - ζ N=2000 dps=50 refinement (full): 700.8s (~11.7 min)
# - ζ N=500 dps=80 refinement: 143.1s -> avg 286 ms/zero
# - ζ N=200 (zeros 501-700) dps=80: 89.4s -> avg 447 ms/zero 
# - L(χ_4 mod 5) N=129 dps=50: 16.7s
# - L(χ_4 mod 5) N=129 dps=80: 44.1s
# 
# De novo comparisons (single zero, mp.zetazero):
# - n=1000, dps=50: 0.43s -> our refinement at n=1000 dps=50 ~ 0.38s (similar)
# - n=1000, dps=80: 0.56s -> our refinement: 0.60s (similar)
# - n=5000, dps=50: 0.82s -> our refinement: 0.81s
# - n=5000, dps=80: 1.34s -> our refinement: 1.28s
# 
# Conclusion: Refinement from Odlyzko seed is ROUGHLY COMPARABLE to de novo mp.zetazero. 
# Speedup ~ 1.0-1.5x, NOT orders of magnitude.
# 
# Build the speedup analysis.

import numpy as np

# Per-zero benchmarks (in seconds), gathered above
bench_n = np.array([1, 100, 1000, 5000])
denovo_50_s = np.array([0.0234, 0.1979, 0.4304, 0.8217])
denovo_80_s = np.array([0.0269, 0.2385, 0.5600, 1.3428])
refine_50_s = np.array([0.0327, 0.1147, 0.3794, 0.8131])
refine_80_s = np.array([0.0548, 0.1543, 0.6029, 1.2832])

# Fit cost ~ k * n^alpha
def fit_loglog(x, y):
 p = np.polyfit(np.log(x), np.log(y), 1)
 return np.exp(p[1]), p[0] # (k, alpha)

ks = {}
for name, y in [('denovo_50', denovo_50_s), ('denovo_80', denovo_80_s),
 ('refine_50', refine_50_s), ('refine_80', refine_80_s)]:
 k, a = fit_loglog(bench_n, y)
 ks[name] = (k, a)
 print(f"{name}: t(n) ≈ {k:.4f} * n^{a:.3f}")

# Total time estimate
def total_time(k, alpha, N):
 # sum of k*n^alpha for n=1..N ≈ integral 1..N of k*n^alpha dn = k * (N^(alpha+1)-1)/(alpha+1)
 return k * (N**(alpha+1)) / (alpha+1)

print(f"\n{'N':>6} {'refine50 (s)':>14} {'denovo50 (s)':>14} {'speedup':>10} {'refine80 (s)':>14} {'denovo80 (s)':>14} {'speedup':>10}")
for N in [2000, 5000, 10000, 20000]:
 tr50 = total_time(*ks['refine_50'], N)
 td50 = total_time(*ks['denovo_50'], N)
 tr80 = total_time(*ks['refine_80'], N)
 td80 = total_time(*ks['denovo_80'], N)
 print(f"{N:>6} {tr50:>14.1f} {td50:>14.1f} {td50/tr50:>10.2f}x {tr80:>14.1f} {td80:>14.1f} {td80/tr80:>10.2f}x")


denovo_50: t(n) ≈ 0.0249 * n^0.417
denovo_80: t(n) ≈ 0.0274 * n^0.452
refine_50: t(n) ≈ 0.0282 * n^0.375
refine_80: t(n) ≈ 0.0447 * n^0.369

 N refine50 (s) denovo50 (s) speedup refine80 (s) denovo80 (s) speedup
 2000 709.3 837.2 1.18x 1082.7 1172.6 1.08x
 5000 2499.8 3066.9 1.23x 3797.0 4436.6 1.17x
 10000 6483.1 8189.3 1.26x 9810.3 12139.5 1.24x
 20000 16813.5 21867.0 1.30x 25346.7 33216.5 1.31x


In [42]:

# Confirmed: refinement-from-Odlyzko offers only a modest 1.1-1.3x speedup vs de novo mp.zetazero.
# This is NOT "orders of magnitude faster" as the hypothesis claims.
# 
# Save the speedup table & generate final figure.

# Validate the actual measured time matches estimate
print(f"Measured N=2000 dps=50: 700.8s, estimated: 709.3s (close)")
print(f"Measured first 500 dps=80: 143.1s")
# Predict refinement of first 500 from fit:
k, a = ks['refine_80']
pred_500 = total_time(k, a, 500)
print(f" predicted: {pred_500:.1f}s")

# Now make summary figure
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(2, 1, figsize=(9, 9))

# Panel A: per-zero time scaling (refinement vs de novo)
ax = axes[0]
n_grid = np.array([1, 100, 1000, 5000])
ax.loglog(n_grid, refine_50_s, 'o-', label='Refine (Odlyzko→Newton), dps=50', color='steelblue')
ax.loglog(n_grid, denovo_50_s, 's--', label='De novo (mp.zetazero), dps=50', color='steelblue', alpha=0.5)
ax.loglog(n_grid, refine_80_s, 'o-', label='Refine, dps=80', color='firebrick')
ax.loglog(n_grid, denovo_80_s, 's--', label='De novo, dps=80', color='firebrick', alpha=0.5)
ax.set_xlabel('zero index n')
ax.set_ylabel('time per zero (s)')
ax.set_title('A. Per-zero refinement cost vs de-novo cost — Riemann ζ')
ax.legend(loc='lower right', fontsize=9)
ax.grid(True, which='both', alpha=0.3)

# Panel B: total time per (N, dps) and speedup factor
ax = axes[1]
Ns = [2000, 5000, 10000, 20000]
ref50 = [total_time(*ks['refine_50'], N)/60 for N in Ns]
den50 = [total_time(*ks['denovo_50'], N)/60 for N in Ns]
ref80 = [total_time(*ks['refine_80'], N)/60 for N in Ns]
den80 = [total_time(*ks['denovo_80'], N)/60 for N in Ns]
x = np.arange(len(Ns))
w = 0.18
ax.bar(x - 1.5*w, ref50, w, label='Refine dps=50', color='steelblue')
ax.bar(x - 0.5*w, den50, w, label='De novo dps=50', color='steelblue', alpha=0.5, hatch='//')
ax.bar(x + 0.5*w, ref80, w, label='Refine dps=80', color='firebrick')
ax.bar(x + 1.5*w, den80, w, label='De novo dps=80', color='firebrick', alpha=0.5, hatch='//')
# Add measured point
ax.scatter([0 - 1.5*w], [700.8/60], color='black', zorder=5, s=80, marker='*', label='Measured (N=2000, dps=50)')
ax.set_xticks(x)
ax.set_xticklabels([str(N) for N in Ns])
ax.set_xlabel('N_zeros')
ax.set_ylabel('Total wall time (min)')
ax.set_title('B. Extrapolated total time per (N, dps) — Refine vs De-novo (ratios 1.08–1.31×)')
ax.legend(fontsize=8, ncol=2, loc='upper left')
ax.grid(True, axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('zeta_refinement_vs_denovo.png', dpi=130, bbox_inches='tight')
plt.close()
print("Saved zeta_refinement_vs_denovo.png")


Measured N=2000 dps=50: 700.8s, estimated: 709.3s (close)
Measured first 500 dps=80: 143.1s
 predicted: 162.2s


Saved zeta_refinement_vs_denovo.png


In [43]:

# Final: write summary report and list artifacts
import os
# List cached files
for root, dirs, files in os.walk('cached_zeros'):
 for f in sorted(files):
 path = os.path.join(root, f)
 print(f"{path}: {os.path.getsize(path)} bytes")
print(f"\ndownloads/odlyzko_zeros1.txt: {os.path.getsize('downloads/odlyzko_zeros1.txt')} bytes")
print(f"downloads/lmfdb_Lchi5.json: {os.path.getsize('downloads/lmfdb_Lchi5.json')} bytes")
print(f"zeta_refinement_vs_denovo.png: {os.path.getsize('zeta_refinement_vs_denovo.png')} bytes")

# Summary stats
print("\n=== SUMMARY ===")
print(f"Odlyzko zeros loaded: {len(odlyzko_zeros)} (precision ~9-10 dp)")
print(f"R2 gate (ζ): PASS (|γ_i - target| < 1e-8 for i=1,2,3)")
print(f"LMFDB L(chi_4 mod 5) zeros loaded: 129 (precision ~30 dp)")
print(f" --> LIMITATION: LMFDB only provides 129 zeros (up to t≈199), not 20,000")
print(f"\nFull refinement N=2000 dps=50: 700.8s ({700.8/60:.2f} min)")
print(f"Sample refinement N=500 dps=80: 143.1s ({143.1/60:.2f} min)")
print(f"L(chi) 129 zeros dps=50: 16.7s; dps=80: 44.1s")
print(f"\nHypothesis test: refinement vs de novo speedup:")
print(f" N=2000, dps=50: refine 11.8 min vs denovo 14.0 min -> 1.18x speedup")
print(f" N=2000, dps=80: refine 18.0 min vs denovo 19.5 min -> 1.08x speedup")
print(f" N=20000, dps=80: refine 422 min vs denovo 554 min -> 1.31x speedup")
print(f" --> Speedup is at best ~1.3x, NOT 'orders of magnitude'. HYPOTHESIS REFUTED for mpmath.")


cached_zeros/Lchi5/N129_dps50.txt: 7241 bytes
cached_zeros/Lchi5/N129_dps80.txt: 11111 bytes
cached_zeros/zeta/N2000_dps50.txt: 108188 bytes
cached_zeros/zeta/N2000_dps80_partial500.txt: 42128 bytes
cached_zeros/zeta/N2000_dps80_partial700.txt: 58928 bytes
cached_zeros/zeta/odlyzko_zeros1_raw.txt: 4346965 bytes

downloads/odlyzko_zeros1.txt: 1800000 bytes
downloads/lmfdb_Lchi5.json: 9803 bytes
zeta_refinement_vs_denovo.png: 143826 bytes

=== SUMMARY ===
Odlyzko zeros loaded: 100000 (precision ~9-10 dp)
R2 gate (ζ): PASS (|γ_i - target| < 1e-8 for i=1,2,3)
LMFDB L(chi_4 mod 5) zeros loaded: 129 (precision ~30 dp)
 --> LIMITATION: LMFDB only provides 129 zeros (up to t≈199), not 20,000

Full refinement N=2000 dps=50: 700.8s (11.68 min)
Sample refinement N=500 dps=80: 143.1s (2.38 min)
L(chi) 129 zeros dps=50: 16.7s; dps=80: 44.1s

Hypothesis test: refinement vs de novo speedup:
 N=2000, dps=50: refine 11.8 min vs denovo 14.0 min -> 1.18x speedup
 N=2000, dps=80: refine 18.0 min vs denovo

In [44]:
from e14c.filesystem import write_file
report = '''<output>
<conclusion>
Importing trusted low-precision zero tables and refining them in mpmath is feasible for ζ and the small LMFDB-provided χ₄ mod 5 list, but it yields only a modest ~1.1–1.3× speedup over de novo `mpmath.zetazero` for ζ, so the hypothesis of orders-of-magnitude acceleration is refuted and the full {2000,5000,10000,20000}×{50,80} grid could not be completed within the compute budget.
</conclusion>
<methods>
I worked in Python under `/workspace/f839811a-5d45-4a51-8681-5f83eaad18f6` using `mpmath`, `numpy`, `matplotlib`, `json`, and standard library I/O/network tools. Analysis plan: (1) download and inspect external zero tables; (2) perform data quality control and validation; (3) refine imported seeds to higher precision; (4) cache outputs to disk; (5) benchmark refinement versus de novo generation; (6) summarize results in a final figure. For the Riemann zeta function, I downloaded Odlyzko’s `zeros1` table (`https://www-users.cse.umn.edu/~odlyzko/zeta_tables/zeros1`), which contains 100,000 imaginary parts of nontrivial zeros at about 9–10 decimal digits; this is sufficient for the required N up to 20,000, although it is not the requested `zeros6.gz`. I validated the imported ζ table against the R2 gate by checking the first three zeros against γ₁=14.134725141, γ₂=21.022039639, γ₃=25.010857580 with tolerance 1e-6. For the Dirichlet control, automated access to bulk LMFDB zeros was limited; I located the LMFDB download endpoint for `L/Character/Dirichlet/5/4`, specifically `/L/download_zeros/1-5-5.4-r0-0-0`, and parsed the provided JSON-like payload. That source returned only 129 positive zeros up to t≈199, so it does not support the target N={2000,5000,10000,20000}. I defined the quadratic character mod 5 as `chi5 = [0,1,-1,-1,1]` and constructed a real Hardy-type Z-function on the critical line for validation and refinement.

For ζ refinement, a naive `findroot(mp.siegelz, seed)` approach was too slow. I therefore implemented an explicit Newton method with precision staging: starting from the imported seed, I performed one Newton update at ~22 digits, one at ~45 digits, and one or two final updates at target precision (`dps+15`) using `mp.siegelz(t)` and `mp.siegelz(t, derivative=1)`. This reduced overhead substantially while preserving accuracy. I fully refined the first 2000 ζ zeros to dps=50 and partially refined the first 700 ζ zeros to dps=80. Refined zeros were saved to canonical cache paths in `cached_zeros/zeta/`. For the χ₄ mod 5 L-function, I refined all 129 imported zeros at dps=50 and dps=80 using Newton’s method on the real Hardy-type Z-function, with numerical differentiation for Z′(t), and cached them in `cached_zeros/Lchi5/`. To compare performance, I benchmarked per-zero wall times at n={1,100,1000,5000} for both refinement-from-seed and de novo `mp.zetazero(n)` at target dps 50 and 80, fit log-log timing models `t(n) ≈ k n^α`, and extrapolated total runtime to N={2000,5000,10000,20000}. I created a final 2-panel figure (`zeta_refinement_vs_denovo.png`) summarizing per-zero scaling and total runtime extrapolations.
</methods>
<results>
Data acquisition and validation: Odlyzko `zeros1` downloaded successfully (100,000 zeros; 1,800,000 bytes raw download). The first three imported ζ zeros were 14.134725142, 21.022039639, and 25.010857580, differing from the R2 targets by 1.0e-9, 0, and 0 respectively, so the ζ import passed the R2 gate to at least 6 decimal places. The LMFDB endpoint for the Dirichlet L-function returned 129 positive zeros with approximately 30 decimal digits; evaluation of the constructed Hardy-type Z-function at the first three imported LMFDB zeros gave residuals on the order of 1e-29 to 1e-30, supporting internal consistency.

Refined caches produced: `cached_zeros/zeta/N2000_dps50.txt` (full 2000 zeros), `cached_zeros/zeta/N2000_dps80_partial500.txt`, `cached_zeros/zeta/N2000_dps80_partial700.txt`, `cached_zeros/zeta/odlyzko_zeros1_raw.txt`, `cached_zeros/Lchi5/N129_dps50.txt`, and `cached_zeros/Lchi5/N129_dps80.txt`. For ζ, full refinement of N=2000 to dps=50 took 700.8 s (11.68 min). The first three refined zeros were γ₁=14.1347251417346937904572519836, γ₂=21.0220396387715549926284795939, γ₃=25.0108575801456887632137909926, with residuals |ζ(1/2+iγ₁)|=3.95e-51, |ζ(1/2+iγ₂)|=2.08e-51, and |ζ(1/2+iγ₃)|=2.37e-50. Partial ζ refinement to dps=80 took 143.1 s for the first 500 zeros and 89.4 s for zeros 501–700; the mean time for the first 500 was 0.286 s/zero. For χ₄ mod 5, refining all 129 imported zeros took 16.73 s at dps=50 and 44.14 s at dps=80. The worst post-refinement residuals were max |L(1/2+iρ,χ)|=3.37e-41 at dps=50 and 1.18e-78 at dps=80.

Benchmark comparison for ζ (per-zero timings, refinement vs de novo `mp.zetazero`): at n=1, refine50=0.0327 s vs denovo50=0.0234 s and refine80=0.0548 s vs denovo80=0.0269 s; at n=100, refine50=0.1147 s vs denovo50=0.1979 s and refine80=0.1543 s vs denovo80=0.2385 s; at n=1000, refine50=0.3794 s vs denovo50=0.4304 s and refine80=0.6029 s vs denovo80=0.5600 s; at n=5000, refine50=0.8131 s vs denovo50=0.8217 s and refine80=1.2832 s vs denovo80=1.3428 s. Fitted timing laws were: de novo dps=50, `t(n) ≈ 0.0249 n^0.417`; de novo dps=80, `t(n) ≈ 0.0274 n^0.452`; refine dps=50, `t(n) ≈ 0.0282 n^0.375`; refine dps=80, `t(n) ≈ 0.0447 n^0.369`. Extrapolated total runtimes (refine vs de novo) were: N=2000, dps=50: 709 s vs 837 s (1.18× speedup); N=2000, dps=80: 1083 s vs 1173 s (1.08×); N=5000, dps=50: 2500 s vs 3067 s (1.23×); N=5000, dps=80: 3797 s vs 4437 s (1.17×); N=10000, dps=50: 6483 s vs 8189 s (1.26×); N=10000, dps=80: 9810 s vs 12140 s (1.24×); N=20000, dps=50: 16814 s vs 21867 s (1.30×); N=20000, dps=80: 25347 s vs 33217 s (1.31×). Thus, refinement from imported ζ seeds improved runtime only modestly and not by orders of magnitude.
</results>
<challenges>
The requested workflow could not be completed exactly as written because the external sources did not support the full design. First, I did not download Odlyzko `zeros6.gz`; instead I used `zeros1`, which was directly accessible and already contains far more than the 20,000 ζ zeros needed. Second, the LMFDB source for the χ₄ mod 5 Dirichlet L-function exposed only 129 zeros, far short of the target N grid, so I cannot generate validated imported-and-refined caches for N={2000,5000,10000,20000} for that control from trusted external tables alone. I cannot complete the full requested χ₄ mod 5 grid because the required imported zero table is not available from the accessed LMFDB endpoint in sufficient length, and generating the missing zeros de novo would no longer test the stated import-and-refine hypothesis. Third, a naive `findroot` implementation for ζ was much slower than expected; only after replacing it with explicit staged-precision Newton iterations did refinement become competitive with `mp.zetazero`. Fourth, the full requested grid `{2000,5000,10000,20000} × {50,80}` for ζ alone remained too expensive for the available runtime, especially at dps=80, so I executed one full cell (N=2000,dps=50), partial dps=80 runs, and statistically grounded extrapolations for the larger cases. These extrapolations matched the measured N=2000,dps=50 runtime closely (predicted 709.3 s vs observed 700.8 s), but they remain model-based estimates rather than direct measurements for the larger cells.
</challenges>
<discussion>
The central hypothesis is not supported in the tested environment. For the Riemann zeta function, importing low-precision Odlyzko tables and refining them with mpmath does help slightly, but the benefit is only incremental because `mpmath.zetazero` is already an efficient specialized de novo routine. The dominant cost at high precision is evaluating the Hardy Z-function and its derivative near large heights, so saving the initial zero-location search step does not reduce runtime by orders of magnitude. In practical terms, the hybrid import-and-refine strategy is still useful for reproducibility and for controlling initial seeds, but it does not solve the computational bottleneck by itself when using mpmath as the refinement backend. For ζ, the best observed gain was around 30% in extrapolation, not 10× or 100×.

For the χ₄ mod 5 Dirichlet L-function, the main issue is data availability rather than refinement cost. The LMFDB-provided list was already high precision and refinement was fast, but the source exposed only 129 zeros. Therefore, the import-and-refine strategy cannot currently populate the full N-grid for this control using the accessed trusted table. This means the project’s foundational cache-building step remains only partially solvable under the present protocol and compute budget. A stronger production workflow would likely mix trusted external tables where available, specialized backends for high-order zero generation where possible, and direct de novo generation only when no table exists. The result also reinforces the dataset note that external data quality and availability constraints materially shape what parts of the computational pipeline are feasible.
</discussion>
<proposed-next-hypotheses>
1. Replacing `mpmath` refinement with a specialized backend (e.g. PARI/GP or compiled Odlyzko-style routines) will reduce the ζ seed-refinement wall time by at least 5× relative to both staged-Newton mpmath refinement and `mpmath.zetazero` at dps=80.
2. For the Dirichlet χ₄ mod 5 control, direct de novo generation of zeros via a validated Hardy-Z sign-change scan and Newton refinement will be more scalable than relying on currently exposed LMFDB zero downloads, enabling at least the N=2000 cache within the same compute budget.
</proposed-next-hypotheses>
<artifacts>
<artifact>
<file-name>downloads/odlyzko_zeros1.txt</file-name>
<artifact-type>external_source</artifact-type>
<artifact-description>Downloaded Odlyzko table containing the first 100,000 nontrivial Riemann zeta zero ordinates at about 9–10 decimal digits. Retrieved directly from the public Odlyzko zeta tables and used as the low-precision seed source for ζ refinement.</artifact-description>
</artifact>
<artifact>
<file-name>downloads/lmfdb_Lchi5.json</file-name>
<artifact-type>external_source</artifact-type>
<artifact-description>Downloaded LMFDB zero list for the Dirichlet L-function associated with character 5.4 (`1-5-5.4-r0-0-0`). The payload contained 129 positive zeros and metadata, which were parsed and used as refinement seeds.</artifact-description>
</artifact>
<artifact>
<file-name>cached_zeros/zeta/odlyzko_zeros1_raw.txt</file-name>
<artifact-type>agent_produced</artifact-type>
<artifact-description>Local cached copy of the imported Odlyzko ζ zero table with provenance header. Created from the downloaded `zeros1` source and stored to support reproducible downstream analyses without repeated downloads.</artifact-description>
</artifact>
<artifact>
<file-name>cached_zeros/zeta/N2000_dps50.txt</file-name>
<artifact-type>agent_produced</artifact-type>
<artifact-description>Refined high-precision cache of the first 2000 nontrivial ζ zero ordinates at 50 decimal digits. Generated from Odlyzko seeds using staged explicit Newton updates on `mpmath.siegelz` and validated by residual checks.</artifact-description>
</artifact>
<artifact>
<file-name>cached_zeros/zeta/N2000_dps80_partial500.txt</file-name>
<artifact-type>agent_produced</artifact-type>
<artifact-description>Partial high-precision cache of the first 500 ζ zero ordinates at 80 decimal digits. Produced to quantify dps=80 runtime scaling within the available compute budget.</artifact-description>
</artifact>
<artifact>
<file-name>cached_zeros/zeta/N2000_dps80_partial700.txt</file-name>
<artifact-type>agent_produced</artifact-type>
<artifact-description>Extended partial cache of the first 700 ζ zero ordinates at 80 decimal digits. Generated by continuing the dps=80 refinement in an additional batch after the initial 500-zero run.</artifact-description>
</artifact>
<artifact>
<file-name>cached_zeros/Lchi5/N129_dps50.txt</file-name>
<artifact-type>agent_produced</artifact-type>
<artifact-description>Refined cache of all 129 imported χ₄ mod 5 Dirichlet L-function positive zeros at 50 decimal digits. Built from LMFDB seeds using Newton updates on a real Hardy-type Z-function.</artifact-description>
</artifact>
<artifact>
<file-name>cached_zeros/Lchi5/N129_dps80.txt</file-name>
<artifact-type>agent_produced</artifact-type>
<artifact-description>Refined cache of all 129 imported χ₄ mod 5 Dirichlet L-function positive zeros at 80 decimal digits. Residual checks showed max |L(1/2+iρ,χ)| ≈ 1.18e-78 after refinement.</artifact-description>
</artifact>
<artifact>
<file-name>zeta_refinement_vs_denovo.png</file-name>
<artifact-type>agent_produced</artifact-type>
<artifact-description>Final 2-panel summary figure comparing per-zero and total extrapolated runtime for imported-seed refinement versus de novo `mpmath.zetazero` generation. Generated with matplotlib from measured benchmark points and log-log timing fits.</artifact-description>
</artifact>
</artifacts>
</output>'''
write_file('analysis_report.xml', report)
print('Wrote analysis_report.xml')

Wrote analysis_report.xml
